# Train SUB-ai 2.0 on Google Colab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/subhobhai943/SUB-ai-2.0/blob/main/train_colab.ipynb)

This notebook fine-tunes the **`Qwen/Qwen2.5-3B-Instruct`** base model (configured in `model/config.py`) with a LoRA adapter on a free **Tesla T4** GPU, then packages the trained adapter for download.

**Before running:** click **Runtime -> Change runtime type** and select **T4 GPU**, then run the cells below in order.

See `docs/COLAB_TRAINING.md` for the full written walkthrough this notebook automates.

## Step 0: Confirm GPU is attached

In [ ]:
!nvidia-smi

## Step 1: Clone the repository

In [ ]:
!git clone https://github.com/subhobhai943/SUB-ai-2.0.git
%cd SUB-ai-2.0

## Step 2: Install dependencies

Installs `transformers`, `peft` (LoRA), `accelerate`, and the rest of `requirements.txt`.

In [ ]:
!pip install -q -r requirements.txt

## Step 3: Download the HuggingFace datasets

Pulls SQuAD, OpenAssistant conversations, TriviaQA, OpenBookQA, Alpaca, CommonsenseQA, GSM8K, and Python code datasets into `data/raw/`.

In [ ]:
!python dataset/import_hf_datasets.py

## Step 4: Assemble the merged training dataset

In [ ]:
!python dataset/build_dataset.py

## Step 5: Train / fine-tune the model

Loads `Qwen/Qwen2.5-3B-Instruct`, wraps it with a LoRA adapter (per `model/config.py`), and fine-tunes in fp16. Best checkpoint (by validation loss) is saved to `checkpoints/pretrained_model/`.

This is the long-running cell -- expect it to take a while depending on dataset size and epoch count.

In [ ]:
!python train.py

## Step 6: Zip and download the fine-tuned adapter weights

In [ ]:
from google.colab import files

!zip -r checkpoints_pretrained.zip checkpoints/pretrained_model checkpoints/config.json
files.download('checkpoints_pretrained.zip')

## Step 7: Run locally

1. Extract `checkpoints_pretrained.zip` into your local `SUB-ai-2.0/` folder so it matches:
   ```text
   SUB-ai-2.0/
   └── checkpoints/
       ├── config.json
       └── pretrained_model/
           ├── adapter_config.json
           ├── adapter_model.safetensors
           ├── tokenizer_config.json
           └── ...
   ```
2. Confirm `model/config.py` locally still has `"model_mode": "pretrained"` and `"pretrained_model_name": "Qwen/Qwen2.5-3B-Instruct"` to match `checkpoints/config.json`.
3. Evaluate: `python evaluate.py`
4. Chat with the fine-tuned model: `python chat.py`